# Walmart Demand Forecasting (OCI AI Foundations Project)

## Project Overview
This project demonstrates an end-to-end demand forecasting workflow using historical Walmart sales data. The goal is to build, evaluate, and improve predictive models that can support inventory planning and supply chain decision-making in a retail environment.

This project is designed to align with:
- Oracle Cloud Infrastructure (OCI) AI Foundations
- OCI Generative AI Professional
- OCI Multicloud Architect Professional

## Business Problem

Retailers must accurately forecast product demand in order to:
- Reduce stockouts
- Avoid overstocking
- Improve supply chain efficiency
- Increase revenue and customer satisfaction

This project explores how machine learning models can be used to predict
weekly sales using historical store-level data and external economic indicators.

## Dataset

**Source:** Public Walmart Store Sales Dataset

**Data includes:**
- Store identifier
- Date
- Weekly sales (target variable)
- Holiday indicator
- Temperature
- Fuel price
- Consumer Price Index (CPI)
- Unemployment rate

## Success Criteria

- Build a reliable baseline forecasting model
- Improve model performance using structured features
- Compare models using RMSE and MAE
- Produce insights relevant to retail supply chain planning

## Step 1: Data Loading and Initial Exploration

The goal of this step is to:
- Load the dataset
- Validate data types and structure
- Perform basic sanity checks before modeling

In [13]:
import pandas as pd

# Load dataset
df = pd.read_csv("Walmart_store_sales.csv")

# Preview data
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [14]:
# Dataset info
df.info()

# Summary statistics
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.000000,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
std,12.988182,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885
min,1.000000,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000


## Step 2: Data Preparation and Problem Framing

In this step we:
- Convert date fields to datetime format
- Sort observations chronologically to prevent data leakage
- Define features and target variables
- Split data using a time-aware approach

In [16]:
# Convert Date column (day-first format)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

# Sort chronologically
df = df.sort_values("Date")

In [17]:
from sklearn.model_selection import train_test_split

# Features and target
X = df[
    ["Store", "Holiday_Flag", "Temperature", "Fuel_Price", "CPI", "Unemployment"]
]
y = df["Weekly_Sales"]

# Time-aware split (no shuffling)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

## Step 3: Baseline Demand Forecasting Model

A baseline model provides a reference point to evaluate whether
more complex machine learning models offer meaningful improvements.
We begin with a linear regression model using structured features.

In [18]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Feature types
categorical_features = ["Store", "Holiday_Flag"]
numerical_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

# Linear Regression pipeline
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [19]:
# Train model
linear_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Predictions
y_pred_lr = linear_model.predict(X_test)

# Metrics
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)

mae_lr, rmse_lr

(75855.90848118925, np.float64(107829.20109546094))

## Linear Regression Results

The baseline linear regression model demonstrates that structured
store-level and economic features can explain a meaningful portion
of weekly sales variability.

This model establishes a performance benchmark for evaluating
more advanced machine learning approaches in subsequent steps.

## Step 4: Improved Demand Forecasting with Random Forest

To improve upon the baseline linear regression model, we introduce a Random Forest Regressor.
Random Forest models are well-suited for demand forecasting because they:
- Capture non-linear relationships in sales data
- Handle interactions between store, holiday, and economic features
- Are robust to noise and outliers

This step evaluates whether a tree-based ensemble model provides meaningful performance improvements over the baseline.

In [22]:
from sklearn.ensemble import RandomForestRegressor

# Define Random Forest model
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [23]:
# Train Random Forest model
rf_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
# Predictions
y_pred_rf = rf_model.predict(X_test)

# Metrics
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)

mae_rf, rmse_rf

print("Random Forest MAE:", mae_rf)
print("Random Forest RMSE:", rmse_rf)

Random Forest MAE: 92573.67346697736
Random Forest RMSE: 133009.2076698084


### Model Performance Comparison

| Model | MAE | RMSE |
|------|-----|------|
| Linear Regression | ~75,856 | ~107,829 |
| Random Forest | (computed above) | (computed above) |

The Random Forest model is expected to outperform the linear baseline by capturing non-linear demand patterns and feature interactions. This demonstrates the value of advanced machine learning techniques in retail demand forecasting.

## Step 4: Model Comparison and Evaluation

Two machine learning models were evaluated for weekly demand forecasting:

- Linear Regression (baseline model)
- Random Forest Regressor (nonlinear ensemble model)

### Evaluation Results

| Model | MAE | RMSE |
|------|-----|------|
| Linear Regression | 75,855.91 | 107,829.20 |
| Random Forest | 92,573.67 | 133,009.21 |

### Key Insights

- The Linear Regression model outperformed the Random Forest model on both MAE and RMSE.
- Despite its greater complexity, the Random Forest model did not improve forecasting accuracy given the current feature set.
- This suggests that structured linear relationships capture demand patterns more effectively than nonlinear models in this configuration.
- The results highlight the importance of feature engineering and time-aware modeling in retail demand forecasting.

Based on these findings, Linear Regression is selected as the preferred model for this iteration.

## Step 5: Feature Engineering

In [26]:
# Create time-based features
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

df[["Date", "Year", "Month", "Week"]].head()

,Date,Year,Month,Week
0,2010-02-05,2010,2,5
429,2010-02-05,2010,2,5
4290,2010-02-05,2010,2,5
2145,2010-02-05,2010,2,5
1430,2010-02-05,2010,2,5


In [27]:
# Define enhanced feature set
X = df[[
    "Store",
    "Holiday_Flag",
    "Year",
    "Month",
    "Week"
]]

y = df["Weekly_Sales"]

# Time-aware train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [28]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Categorical and numerical features
categorical_features = ["Store"]
numerical_features = ["Holiday_Flag", "Year", "Month", "Week"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

# Updated Linear Regression model
enhanced_lr_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

# Train model
enhanced_lr_model.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Predictions
y_pred_enhanced = enhanced_lr_model.predict(X_test)

# Evaluation
mae_enhanced = mean_absolute_error(y_test, y_pred_enhanced)
rmse_enhanced = mean_squared_error(y_test, y_pred_enhanced) ** 0.5

mae_enhanced, rmse_enhanced


(78934.1650829105, 112502.58573741479)

### Feature Engineering Results

Time-based features (year, month, and week) were added to improve demand forecasting performance.

Key observations:
- Incorporating temporal structure improved the model’s ability to capture seasonality.
- Linear Regression continued to perform competitively with minimal computational overhead.
- The approach aligns with production forecasting constraints in retail and supply chain environments.

This step demonstrates how thoughtful feature engineering can outperform more complex models without increasing infrastructure costs.


Lastly, the feature engineering did not improve model performance. The enhanced linear regression model showed higher MAE and RMSE compared to the baseline, suggesting the additional features did not contribute meaningful predictive power for demand forecasting in this dataset.

## Step 6: Final Model Comparison and Conclusion
Model Performance Summary
Three models were evaluated for the Walmart demand forecasting task using Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE):

Baseline Linear Regression
- MAE: 75,855.91
- RMSE: 107,829.20

Random Forest Regressor
- MAE: 92,573.67
- RMSE: 133,009.21

Enhanced Linear Regression (Feature Engineering Applied)
- MAE: 78,934.17
- RMSE: 112,502.59

Lower values indicate better predictive performance for both metrics.

Best Performing Model:

The Baseline Linear Regression model achieved the lowest MAE and RMSE among all evaluated models. Based on these results, it was the most accurate and reliable model for this dataset.

Although more complex models were explored, increased complexity did not lead to improved performance.

Key Observations:
- Feature engineering did not improve model accuracy in this case. The enhanced linear regression model showed higher error metrics than the baseline.

- The Random Forest model underperformed relative to linear regression, suggesting that the dataset may not contain strong nonlinear relationships or that additional tuning would be required.

- Simpler models can outperform more complex ones when the underlying data relationships are largely linear.

Limitations:
- The dataset may lack sufficient feature richness to benefit from nonlinear models.

- Hyperparameter tuning was limited and could potentially improve Random Forest performance.

- External factors affecting demand (e.g., promotions, regional trends, economic conditions) were not included.

In Conclusion, this project demonstrates an end-to-end machine learning workflow, including data preprocessing, model training, evaluation, and comparison. The results highlight the importance of empirical evaluation rather than assuming that more complex models will perform better. For this demand forecasting task, a simple baseline model proved to be the most effective.